# 🏆 Tactical Insights from World Cup 2022

Deep dive into tactical patterns and team-specific insights.

---

## 📦 Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loaders import load_wyscout_data
from src.data.extractors import extract_set_pieces, calculate_success_metrics
from src.features.spatial import calculate_spatial_features
from src.features.temporal import calculate_temporal_features
from src.visualization.pitch import draw_pitch, plot_heatmap, PITCH_LENGTH, PITCH_WIDTH

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Ready for analysis!")

In [ ]:
# Load and prepare data
data = load_wyscout_data()
set_pieces = extract_set_pieces(data)
set_pieces = calculate_spatial_features(set_pieces)
set_pieces = calculate_temporal_features(set_pieces)

print(f"Loaded {len(set_pieces)} set pieces from {set_pieces['team'].nunique()} teams")

## 🏆 Which Teams Were Best at Set Pieces?

In [ ]:
# Team performance analysis
team_stats = set_pieces.groupby('team').agg({
    'event_id': 'count',
    'outcome': [
        lambda x: (x == 'goal').sum(),
        lambda x: (x.isin(['goal', 'shot'])).sum(),
        lambda x: (x.isin(['goal', 'shot'])).mean() * 100
    ]
}).reset_index()

team_stats.columns = ['Team', 'Total SP', 'Goals', 'Shots', 'Success Rate']
team_stats['Goal Rate'] = team_stats['Goals'] / team_stats['Total SP'] * 100
team_stats = team_stats.sort_values('Success Rate', ascending=False)

print("🏆 Top 10 Teams by Set Piece Success Rate:\n")
print(team_stats.head(10).to_string(index=False))

In [ ]:
# Visualize top teams
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By total set pieces
top_10_volume = team_stats.nlargest(10, 'Total SP')
axes[0].barh(top_10_volume['Team'], top_10_volume['Total SP'], color='steelblue')
axes[0].set_xlabel('Total Set Pieces')
axes[0].set_title('Top 10 by Volume', fontweight='bold')
axes[0].invert_yaxis()

# By success rate
top_10_success = team_stats.nlargest(10, 'Success Rate')
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.3, 10))
axes[1].barh(top_10_success['Team'], top_10_success['Success Rate'], color=colors)
axes[1].set_xlabel('Success Rate (%)')
axes[1].set_title('Top 10 by Success Rate', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## ⚽ What Makes a Successful Corner?

In [ ]:
# Filter to corners only
corners = set_pieces[set_pieces['type'] == 'corner'].copy()
print(f"Analyzing {len(corners)} corners...")

# Success by delivery type
if 'delivery_type' in corners.columns:
    delivery_success = corners.groupby('delivery_type').agg({
        'event_id': 'count',
        'outcome': lambda x: (x.isin(['goal', 'shot'])).mean() * 100
    }).reset_index()
    delivery_success.columns = ['Delivery Type', 'Count', 'Success Rate']
    delivery_success = delivery_success.sort_values('Success Rate', ascending=False)
    
    print("\n📊 Corner Success by Delivery Type:\n")
    print(delivery_success.to_string(index=False))

In [ ]:
# Visualize delivery effectiveness
if 'delivery_type' in corners.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(delivery_success)))
    bars = ax.bar(delivery_success['Delivery Type'], delivery_success['Success Rate'], color=colors)
    
    ax.set_ylabel('Success Rate (%)')
    ax.set_title('Corner Kick Success by Delivery Type', fontweight='bold')
    
    for bar, rate in zip(bars, delivery_success['Success Rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{rate:.1f}%', ha='center', fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Success by number of attackers
corners['success'] = corners['outcome'].isin(['goal', 'shot'])
att_success = corners.groupby('n_attackers_in_box')['success'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
att_success.plot(kind='bar', ax=ax, color='coral')
ax.set_xlabel('Number of Attackers in Box')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Corner Success by Number of Attackers', fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 Optimal Positioning Analysis

In [ ]:
# Heatmap for successful vs unsuccessful corners
successful_corners = corners[corners['outcome'].isin(['goal', 'shot'])]
unsuccessful_corners = corners[~corners['outcome'].isin(['goal', 'shot'])]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Successful
_, ax1 = plot_heatmap(successful_corners, zone_type='receiver', ax=axes[0])
axes[0].set_title('Successful Corners', fontweight='bold')

# Unsuccessful
_, ax2 = plot_heatmap(unsuccessful_corners, zone_type='receiver', ax=axes[1])
axes[1].set_title('Unsuccessful Corners', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Danger index analysis
if 'danger_index' in corners.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.hist(unsuccessful_corners['danger_index'], bins=20, alpha=0.5, label='Unsuccessful', color='red')
    ax.hist(successful_corners['danger_index'], bins=20, alpha=0.5, label='Successful', color='green')
    
    ax.set_xlabel('Danger Index')
    ax.set_ylabel('Frequency')
    ax.set_title('Danger Index: Successful vs Unsuccessful Corners', fontweight='bold')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMean Danger Index:")
    print(f"  Successful: {successful_corners['danger_index'].mean():.1f}")
    print(f"  Unsuccessful: {unsuccessful_corners['danger_index'].mean():.1f}")

## 🛡️ Defensive Strategies Analysis

In [ ]:
# Defensive success (clearances and possession won)
defensive_outcomes = set_pieces.groupby('n_defenders_in_box').agg({
    'event_id': 'count',
    'outcome': [
        lambda x: (x == 'clearance').mean() * 100,
        lambda x: (x == 'possession_lost').mean() * 100,  # Attacking team lost = defense won
    ]
}).reset_index()

defensive_outcomes.columns = ['Defenders', 'Count', 'Clearance Rate', 'Defense Success']

fig, ax = plt.subplots(figsize=(10, 5))
x = defensive_outcomes['Defenders']
ax.plot(x, defensive_outcomes['Defense Success'], 'o-', color='blue', label='Defense Won Possession')
ax.plot(x, defensive_outcomes['Clearance Rate'], 's-', color='green', label='Clearance Rate')

ax.set_xlabel('Number of Defenders in Box')
ax.set_ylabel('Rate (%)')
ax.set_title('Defensive Effectiveness by Number of Defenders', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## ⏰ Temporal Patterns

In [ ]:
# Success rate by game phase
if 'game_phase_numeric' in set_pieces.columns:
    phase_labels = ['0-15', '15-30', '30-45', '45-60', '60-75', '75-90']
    
    phase_success = set_pieces.groupby('game_phase_numeric').agg({
        'event_id': 'count',
        'outcome': lambda x: (x.isin(['goal', 'shot'])).mean() * 100
    }).reset_index()
    phase_success.columns = ['Phase', 'Count', 'Success Rate']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(range(len(phase_success)), phase_success['Success Rate'], 
                  color=plt.cm.RdYlGn(np.linspace(0.3, 0.8, len(phase_success))))
    ax.set_xticks(range(len(phase_labels)))
    ax.set_xticklabels(phase_labels)
    ax.set_xlabel('Game Minute')
    ax.set_ylabel('Success Rate (%)')
    ax.set_title('Set Piece Success by Game Phase', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Fatigue impact
if 'fatigue_factor' in set_pieces.columns:
    set_pieces['fatigue_bin'] = pd.cut(set_pieces['fatigue_factor'], bins=5, 
                                        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
    
    fatigue_success = set_pieces.groupby('fatigue_bin').agg({
        'outcome': lambda x: (x.isin(['goal', 'shot'])).mean() * 100
    }).reset_index()
    
    print("\n📊 Success Rate by Fatigue Level:\n")
    print(fatigue_success.to_string(index=False))

## 🎯 Key Tactical Recommendations

Based on our analysis:

### For Attacking Teams:
1. **Use inswingers for corners** - Higher success rate
2. **5-6 attackers optimal** - Diminishing returns beyond
3. **Target near post** - Higher conversion zone
4. **Late game opportunities** - Fatigue creates chances

### For Defending Teams:
1. **7+ defenders for corners** - Reduces success significantly
2. **Zone marking effective** - Better than man-to-man for volume
3. **Quick clearances** - Reduce second ball chances
4. **Maintain concentration late** - More dangerous period

---

**Analysis complete! 🏆**